In [ ]:
import numpy as np
from  numpy import deg2rad as d2r
from  numpy import array as arr

from scipy.optimize import minimize, approx_fprime
from space_traj_opt.models3d import dynamics
from space_traj_opt.controller3d import  CtrlMode

from space_traj_opt.transcription import MultiShootingTranscription
from space_traj_opt.utils import unpack_sol_list
from space_traj_opt.plotting import plot, visualize_jac2

STANDARD_GRAV = 9.80665

## Electron Rocket Parameters

In [ ]:
n_engines_s1 = 9
n_engines_s2 = 1
isp_s1 = 311.0
engine_thrust_s1 = n_engines_s1*24910.04  # N Average between sl and vac
isp_s2 = 343.0
engine_thrust_s2 = n_engines_s2* 25_000.0  # N
s1_vch_params = (engine_thrust_s1, isp_s1)
s2_vch_params = (engine_thrust_s2, isp_s2)

fairing_mass = 50.0
farinig_timing = 184.0 - 162.0 # sec
payload = 250.0
s1_dry_mass = 1076.47308279  
s2_dry_mass = 257.90093739  

s1_wet_mass = 10047.082106064723
s2_wet_mass = 2602.454913676189
total_mass = 12949.537019740912

mdot_s1 = engine_thrust_s1 / STANDARD_GRAV / isp_s1
mdot_s2 = engine_thrust_s2 / STANDARD_GRAV / isp_s2

In [ ]:
NUM_X= 7
NUM_U = 4
NUM_PHASE = 1
# %load_ext snakeviz

## Initial Guesses

In [ ]:
mu_earth = 3.986004418e14
earth_r = 6_378_000.0 # m
circ_orbit_alt = 200_000.0 
v_circ = np.sqrt(mu_earth / (earth_r + circ_orbit_alt))

# Guesses 
s2_sep_mass = s2_wet_mass + payload + fairing_mass
x0 = arr([
    [45000+ earth_r,80000+earth_r,0,2800,1000,0, s2_wet_mass + payload - farinig_timing * mdot_s2]
])


## normalization vector 
x0_n_vec = arr([earth_r, earth_r, earth_r, 5000, 1000, 1000, 5000])

## Define a multiphase trajectory problem

In [ ]:
problem = MultiShootingTranscription(["phase0"], NUM_X, dynamics)

problem.set_dynamics_params("phase0", s1_vch_params)


## Define state, control and time guesses for each phase 

In [ ]:
state_bounds  = [(0, None), (0, None), (0, None),(0, None),(0, None), (0, None), (100, None)]

problem.set_phase_init_x("phase0", x0 = x0[0], norm_vec = x0_n_vec, bounds = state_bounds)
problem.set_phase_control(
    "phase0", 
    CtrlMode.LTS, 
    u0 = arr([-0.001,1, 0,0]), bounds = [(-0.1,0.1), (-3,3), (-0.1,0.1), (-3,3)], norm_vec =[0.1,np.pi/2, 0.1,np.pi/2])
problem.set_phase_time("phase0", t0 = 320)

a_desired = circ_orbit_alt + earth_r
e_desired = 0.0

x_f = arr([a_desired, e_desired, s2_dry_mass + payload])
xf_n_vec = arr([1000, 0.001, 5000])
problem.set_terminal_state(x_final = x_f, bounds = arr([a_desired, e_desired, None]), norm_vec = xf_n_vec)

## Build The problem
Builds the decision vector and bounds 

In [ ]:
d0, d_bounds, normalization_vec, full_params = problem.build()
d0_norm, d_bounds_norm = problem.normalize_decision_vec(d0, d_bounds,normalization_vec)

In [ ]:
problem

In [ ]:
d0

In [ ]:
full_params[0]

In [ ]:
config = full_params[0]
u, x, t_terminal, control_law = problem.unpack_decision_var(d0, config=config)


In [ ]:
u, x, t_terminal

In [ ]:

# make inputs hashable, needed for lru cache, the copy is cheaper than a second f(x) eval
u_ = tuple(u.tolist())
x_ = tuple(x.tolist())
t_ = float(t_terminal)
vch_params = (config[3], (control_law, u_))
#problem.traj_rollout(t_, x_, vch_params)

In [ ]:
np.linspace(0.0, t_,25)

In [ ]:
print(u_, x_, t_, vch_params)

In [ ]:
from scipy.integrate import solve_ivp

sol = solve_ivp(
    dynamics, 
    t_span=[0.0, t_], 
    t_eval= np.linspace(0.0, t_,25),
    y0=x_,    
    args=(vch_params,)
)

In [ ]:
x_

In [ ]:
plot(
    sol.t, sol.y[1]-earth_r,
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pos y",
    )

In [ ]:
circ_orbit_alt, earth_r

## Defining dynamic constraint function

In [ ]:
from space_traj_opt.math.orbital_calcs import rv_to_aei


def dynamics_knot_constrant(decision_var, config_list): 
    """Integrate the dynamics of each segment. Calcculate the defect  between the knot points.
    This vector is used as the equality constraint for the optimization problem.
    The defect is calculated as the difference between the final state of the previous segment and the initial state of the next segment.

    Args:
        decision_var : Optimzation decission vector
        config_list : List of configs for each phase

    Returns:
        Knot defect vector
    """
    d0 = problem.denormalize_decision_vec(decision_var, normalization_vec)
    defect_vector_list = []
    sol_list=  problem.full_traj_rollout(d0, config_list)
    for idx in range(1,NUM_PHASE):
        _,_, knot_defect,_ = config_list[idx]
        defect_sub_vector = sol_list[idx].y[:,0] - sol_list[idx-1].y[:,-1] + knot_defect
        defect_sub_vector /= arr([10000, 10000,10000, 8000, 5000, 5000, 1000])# defect vector normalization
        defect_vector_list.append(defect_sub_vector)
    
    # Terminal Defect
    # calculate orbital elements here

    a_scale =  1000
    e_scale = 0.001
    r = sol_list[-1].y[:,-1][:3]
    v = sol_list[-1].y[:,-1][3:6]
    a, e, i = rv_to_aei(r, v, mu_earth)
    
    terminal_defect = np.array([
        (a   - a_desired) / a_scale,
        (e   - 0) / e_scale,
        #(i   - i_desired) / i_scale,
    ])
    defect_vector_list.append(terminal_defect)
    defect_vec = arr(defect_vector_list).flatten()
    return defect_vec



In [ ]:
dynamics_knot_constrant(d0_norm, full_params)

In [ ]:
constraints = [{'type': 'eq', 'fun': dynamics_knot_constrant, 'args':(full_params,) },]

## Objective function
Maximize stage 2 mass

In [ ]:
def objective(decision_var: tuple, params: tuple) -> float:
    """Objective function for min prop

    Args:
        decision_var : Optimization problem decision vector
        params : 

    Returns:
        Cost to minimize
    """
    terminal_mass= decision_var[-1]
    print("terminal_mass", terminal_mass)
    return -terminal_mass*terminal_mass*10000


def jac_objective(decision_var: tuple, params: tuple):
    """Jac of the decision vector wrt the cost."""
    
    jac = np.zeros_like(decision_var)
    val = -decision_var[-1] - decision_var[-1]
    jac[-1]= val*10000
    return jac

## Scipy Minimize
SLSQP has to be used here because it can handle bounds and equality constraints.

In [ ]:
# %%snakeviz

result = minimize(
    objective, 
    d0_norm, 
    jac= jac_objective,
    method='SLSQP', 
    bounds=d_bounds_norm, 
    constraints=constraints,
    options = {"maxiter": 500, "disp": True},
    args=(full_params,)
)

In [ ]:
constraint_jac = approx_fprime(
    result.x, 
    dynamics_knot_constrant, 
    np.float64(1.4901161193847656e-08), full_params)

In [ ]:
visualize_jac2(result.x, constraint_jac)

In [ ]:
x_opt = problem.denormalize_decision_vec(result.x, normalization_vec)

sol_list = problem.full_traj_rollout(x_opt, full_params)

In [ ]:
plot(
    *unpack_sol_list(sol_list,0),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pos x",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )

In [ ]:
plot(
    *unpack_sol_list(sol_list,1),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pos y",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )


In [ ]:
plot(
    *unpack_sol_list(sol_list,2),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Vel",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )

In [ ]:
plot(
    *unpack_sol_list(sol_list,3),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Vel",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )

In [ ]:
plot(
    *unpack_sol_list(sol_list,4),
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Mass",
    trace_names=("phase0", "phase1", "phase2", "phase3")
    )

In [ ]:
times, state= unpack_sol_list(sol_list,4)

In [ ]:
lts_2 =lts_control(times[2] - times[2][0], 0, problem.unpack_decision_var(x_opt, full_params[2] )[0])
lts_3 =lts_control(times[3]- times[3][0], 0, problem.unpack_decision_var(x_opt, full_params[3] )[0])


In [ ]:
plot(
    [times[2], times[3]],
    [lts_2, lts_3],
    title="Time vs States", 
    xlabel="Time", 
    ylabel="Pitch",
    trace_names=("phase2", "phase3")
    )